In [1]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))


2025-08-21 13:32:05.890926: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-21 13:32:06.922670: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755761527.189389    1090 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755761527.253105    1090 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755761527.932059    1090 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

2.19.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import os
import numpy as np
import tensorflow as tf
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB5
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import RMSprop, Adam, SGD
from tensorflow.keras.applications.efficientnet import preprocess_input
from itertools import product
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

# BPHE preprocessing function
def bphe_rgb(img):
    if img.max() <= 1.0:
        img = (img * 255).astype(np.uint8)
    else:
        img = img.astype(np.uint8)

    ycrcb = cv2.cvtColor(img, cv2.COLOR_RGB2YCrCb)
    y, cr, cb = cv2.split(ycrcb)
    y_eq = cv2.equalizeHist(y)
    ycrcb_eq = cv2.merge((y_eq, cr, cb))
    img_eq = cv2.cvtColor(ycrcb_eq, cv2.COLOR_YCrCb2RGB)

    return preprocess_input(img_eq.astype(np.float32))

# Directories
filtered_train_dir = '/mnt/d/FA019/new/SplittedDataset/train'
filtered_val_dir = '/mnt/d/FA019/new/SplittedDataset/val'

# Parameters
img_size = (456, 456)  # Updated for EfficientNetB7
num_classes = len(os.listdir(filtered_train_dir))

# Data Generators
train_datagen = ImageDataGenerator(
    preprocessing_function=bphe_rgb,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest')

val_datagen = ImageDataGenerator(preprocessing_function=bphe_rgb)

train_generator = train_datagen.flow_from_directory(
    filtered_train_dir, target_size=img_size, batch_size=32, class_mode='categorical')

val_generator = val_datagen.flow_from_directory(
    filtered_val_dir, target_size=img_size, batch_size=32, class_mode='categorical')

# Define function to create model
def create_model(learning_rate, dropout_rate, optimizer):
    base_model = EfficientNetB5(weights='imagenet', include_top=False, input_shape=(img_size[0], img_size[1], 3))
    for layer in base_model.layers[:-30]:
        layer.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='swish')(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(512, activation='swish')(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(1024, activation='swish')(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(512, activation='swish')(x)
    x = Dropout(dropout_rate)(x)
    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)

    if optimizer == 'adam':
        opt = Adam(learning_rate=learning_rate)
    elif optimizer == 'rmsprop':
        opt = RMSprop(learning_rate=learning_rate)
    elif optimizer == 'sgd':
        opt = SGD(learning_rate=learning_rate, momentum=0.9)

    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Define hyperparameter grid
param_grid = {
    'learning_rate': [ 0.0001, 0.001, 0.0003],
    'dropout_rate': [0.5, 0.3],
    'optimizer': ['rmsprop', 'adam', 'sgd'],
    'batch_size': [16, 32]
}

# Perform manual grid search
best_acc = 0
best_params = None

for lr, dropout, opt, batch_size in product(param_grid['learning_rate'], param_grid['dropout_rate'], param_grid['optimizer'], param_grid['batch_size']):
    print(f"\nTraining with lr={lr}, dropout={dropout}, optimizer={opt}, batch_size={batch_size}")
    
    # Recreate train_generator with updated batch size
    train_generator = train_datagen.flow_from_directory(
        filtered_train_dir, target_size=img_size, batch_size=batch_size, class_mode='categorical')
    
    val_generator = val_datagen.flow_from_directory(
        filtered_val_dir, target_size=img_size, batch_size=batch_size, class_mode='categorical')

    model = create_model(lr, dropout, opt)

    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=100,
        verbose=1,
        #workers=2,  # default is often too high
        #max_queue_size=10,
        #use_multiprocessing=False
    )

    val_acc = max(history.history['val_accuracy'])
    print(f"Validation Accuracy: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        best_params = {
            'learning_rate': lr,
            'dropout_rate': dropout,
            'optimizer': opt,
            'batch_size': batch_size
        }

print("\nBest Hyperparameters:", best_params)

Found 1614 images belonging to 5 classes.
Found 229 images belonging to 5 classes.

Training with lr=0.0001, dropout=0.5, optimizer=rmsprop, batch_size=16
Found 1614 images belonging to 5 classes.
Found 229 images belonging to 5 classes.


I0000 00:00:1755761570.132210    1090 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21458 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9
/mnt/d/FA011/tf_gpu_env/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/100


I0000 00:00:1755761585.345961    1507 service.cc:152] XLA service 0x7ce390014740 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1755761585.345987    1507 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2025-08-21 13:33:05.970480: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1755761610.697890    1507 cuda_dnn.cc:529] Loaded cuDNN version 90701
2025-08-21 13:34:10.909760: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_29771', 4 bytes spill stores, 4 bytes spill loads

2025-08-21 13:34:10.927912: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_29771'